# Train v2 – reference-anchored MDD (Iqra'Eval 2025 data only)

**Settings (right panel):** Accelerator = *GPU T4 x2*, Internet = *On*.

**Secrets (Add-ons → Secrets), attach both to this notebook:**
- `HF_TOKEN` – a Hugging Face token with **write** access (checkpoints are saved to a private model repo)
- `WANDB_API_KEY` – from wandb.ai/authorize (optional; logging is skipped without it)

Run with **Save Version → Save & Run All (Commit)** – it runs in the background for up to 12 h.
Checkpoints go to the private repo `HUB_REPO` every 2000 steps and when the session stops (~11.3 h).
**To continue training, just run the notebook again** – it resumes from the Hub automatically and keeps logging to the same wandb run.
To start a fresh experiment, change `HUB_REPO` (and `RUN_NAME`).

In [ ]:
HUB_REPO = "Haitam03/iqraeval-mdd-v2"     # private HF model repo for checkpoints (created automatically)
WANDB_PROJECT = "iqraeval-mdd"
RUN_NAME = "v2-run1"
MAX_STEPS = 30000
BRANCH = "v2"


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    WANDB_ARGS = f"--wandb-project {WANDB_PROJECT} --wandb-name {RUN_NAME}"
except Exception:
    print("no WANDB_API_KEY secret - training without wandb")
    WANDB_ARGS = ""
!rm -rf repo && git clone -q -b $BRANCH https://github.com/hBouanane/wav2vec2-mispronunciation-detector repo
!git -C repo log --oneline -1
!git clone -q --depth 1 https://github.com/Iqra-Eval/interspeech_IqraEval
!pip install -q jiwer "datasets>=3.0,<4" soundfile librosa wandb
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
RUN = "/kaggle/working/run"
!cd repo && PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 v2/train_v2.py     --out $RUN --mdd-eval ../interspeech_IqraEval/mdd_eval --max-steps $MAX_STEPS     --hub-repo $HUB_REPO $WANDB_ARGS 2>&1 | grep --line-buffered -v -i warn

In [ ]:
!cat $RUN/best.json 2>/dev/null | head -5; tail -3 $RUN/eval_log.jsonl 2>/dev/null
!du -sh $RUN/*